### LEVEL 6 - ADVANCED AGENTIC AI WITH REFLECTION

In [1]:
import pandas as pd
import json
import os
from datetime import datetime

print("✅ Level 6 - Advanced Agent with Reflection Initialized")

✅ Level 6 - Advanced Agent with Reflection Initialized


In [2]:
# Load dataset
df = pd.read_excel('C:/Users/HP/Projects/Gen AI Powered Data Analytics/Final_Dataset_with_Agent_Level5_Tools.xlsx')

# Load memory
memory_file = 'logs/customer_memory_with_tools.json'
if os.path.exists(memory_file):
    with open(memory_file, 'r') as f:
        customer_memory = json.load(f)
    print(f"✅ Loaded memory for {len(customer_memory)} customers")
else:
    customer_memory = {}
    print("ℹ️ Starting with fresh memory")

print(f"Total customers: {len(df)}")

✅ Loaded memory for 500 customers
Total customers: 500


In [8]:
 def calculate_risk_score(row):
    """Calculate risk score (0-100) based on key features"""
    score = 0
    
    # Missed Payments - strongest factor
    if row['Missed_Payments'] >= 4:
        score += 50
    elif row['Missed_Payments'] >= 2:
        score += 25
    
    # Credit Utilization
    if row['Credit_Utilization'] > 0.70:
        score += 30
    elif row['Credit_Utilization'] > 0.50:
        score += 15
    
    # Credit Score (lower = higher risk)
    if row['Credit_Score'] < 500:
        score += 15
    elif row['Credit_Score'] < 600:
        score += 8
    
    return min(score, 100)

In [10]:
def send_sms_reminder(customer_row):
    """Tool 1: Send a gentle SMS reminder"""
    print(f"📱 SMS sent to {customer_row['Customer_ID']}: 'Hi, we noticed a missed payment. Please pay at your earliest convenience.'")
    return "SMS_SENT"

def offer_payment_plan(customer_row):
    """Tool 2: Offer a structured payment plan"""
    print(f"📧 Payment plan offered to {customer_row['Customer_ID']}: Flexible installments suggested.")
    return "PAYMENT_PLAN_OFFERED"

def escalate_to_human(customer_row):
    """Tool 3: Escalate to human collections officer"""
    print(f"👤 Escalated {customer_row['Customer_ID']} to human collections team for immediate follow-up.")
    return "ESCALATED_TO_HUMAN"

def update_customer_status(customer_row, new_status):
    """Tool 4: Update customer record"""
    print(f"📊 Updated status for {customer_row['Customer_ID']} to: {new_status}")
    return "STATUS_UPDATED"

def log_action(customer_row, action_taken):
    """Tool 5: Log the action taken"""
    print(f"📝 Logged action '{action_taken}' for customer {customer_row['Customer_ID']}")
    return "LOGGED"

In [11]:
tools = {
    "SMS_REMINDER": send_sms_reminder,
    "PAYMENT_PLAN": offer_payment_plan,
    "ESCALATE": escalate_to_human,
    "MONITOR": lambda x: "MONITORING_CONTINUED"   # No action needed
}

def select_and_execute_tool(customer_row, decision):
    """
    Selects the appropriate tool based on the decision and executes it.
    """
    tool_function = tools.get(decision, log_action)
    
    if decision == "MONITOR":
        result = "MONITORING_CONTINUED"
    elif decision == "ESCALATE":
        result = tool_function(customer_row)
    elif decision == "PAYMENT_PLAN":
        result = tool_function(customer_row)
    else:  # SMS_REMINDER
        result = tool_function(customer_row)
    
    # Log the action
    log_action(customer_row, decision)
    
    return result

##### 3: Reflection Function (New Feature)

In [3]:
# Code Block 3: Reflection Function - Agent learns from past actions
def reflect_on_customer(cust_id, memory):
    """
    Level 6: Agent reflects on previous actions and outcomes.
    """
    history = memory.get(cust_id, {"interactions": []})
    
    if len(history["interactions"]) == 0:
        return "No previous history"
    
    last_action = history["interactions"][-1]["action"]
    last_reason = history["interactions"][-1]["reason"]
    
    reflection = f"Previously took action '{last_action}' because: {last_reason}. "
    
    if len(history["interactions"]) >= 2:
        second_last = history["interactions"][-2]["action"]
        reflection += f"Before that, action was '{second_last}'. "
    
    return reflection

##### 4: Advanced Reasoning with Reflection

In [4]:
# Code Block 4: Advanced Reasoning with Reflection
def advanced_reasoning_with_reflection(customer_row, memory):
    cust_id = customer_row['Customer_ID']
    risk_score = calculate_risk_score(customer_row)
    reflection = reflect_on_customer(cust_id, memory)
    
    # Enhanced decision making with reflection
    if risk_score >= 80:
        decision = "ESCALATE"
        reason = f"Critical risk. {reflection} Immediate escalation is necessary."
    elif risk_score >= 60:
        decision = "PAYMENT_PLAN"
        reason = f"High risk. {reflection} Offering payment plan is the best empathetic approach."
    elif risk_score >= 35:
        decision = "SMS_REMINDER"
        reason = f"Moderate risk. {reflection} A personalized reminder should be effective now."
    else:
        decision = "MONITOR"
        reason = f"Low risk. {reflection} Continued monitoring is sufficient."
    
    # Save to memory
    history = memory.get(cust_id, {"interactions": []})
    history["interactions"].append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "risk_score": risk_score,
        "action": decision,
        "reason": reason
    })
    memory[cust_id] = history
    
    return decision, reason, risk_score

##### 5: Enhanced Guardrails for Level 6

In [5]:
# Code Block 5: Enhanced Guardrails with Reflection
def apply_advanced_guardrails(customer_row, decision, reflection):
    """
    Level 6: More intelligent guardrails that consider reflection.
    """
    age = customer_row.get('Age', 30)
    employment = str(customer_row.get('Employment_Status', '')).lower()
    missed = customer_row.get('Missed_Payments', 0)
    
    guardrail_reason = "No guardrail triggered"
    
    # Strong protection for vulnerable customers
    if age >= 65 or 'retired' in employment or 'unemployed' in employment:
        if decision == "ESCALATE":
            return "PAYMENT_PLAN", "Guardrail: Softened ESCALATE for vulnerable customer"
        elif decision == "PAYMENT_PLAN":
            return "SMS_REMINDER", "Guardrail: Further softened for vulnerable customer"
    
    # First-time or very low missed payments
    if missed <= 1 and decision in ["ESCALATE", "PAYMENT_PLAN"]:
        return "SMS_REMINDER", "Guardrail: First-time offender - using gentle reminder"
    
    # Young customers
    if age < 22 and decision == "ESCALATE":
        return "PAYMENT_PLAN", "Guardrail: Young customer - action softened"
    
    return decision, guardrail_reason

##### 6: Main Advanced Agent Runner (With Tools + Reflection)

In [6]:
# Code Block 6: Main Advanced Agent with Tools & Reflection
def run_advanced_agent(customer_row):
    """
    Full Level 6 Agent: Reasoning with Reflection → Guardrails → Tool Execution
    """
    cust_id = customer_row['Customer_ID']
    
    # Step 1: Advanced Reasoning with Reflection
    decision, reason, risk_score = advanced_reasoning_with_reflection(customer_row, customer_memory)
    
    # Step 2: Apply Guardrails
    final_action, guardrail_reason = apply_advanced_guardrails(customer_row, decision, None)
    
    # Step 3: Execute the Tool
    tool_result = select_and_execute_tool(customer_row, final_action)
    
    # Step 4: Create detailed log
    log = {
        'Customer_ID': cust_id,
        'Risk_Score': risk_score,
        'Initial_Decision': decision,
        'Final_Action': final_action,
        'Reason': reason,
        'Guardrail_Applied': guardrail_reason,
        'Tool_Result': tool_result,
        'Timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    return log, final_action

##### 7: Test the Advanced Agent

In [12]:
# Code Block 7: Test Level 6 Advanced Agent
print("🚀 Testing Level 6 - Advanced Agent with Reflection & Tools\n")

test_customers = df.sample(6, random_state=42)

for idx, customer in test_customers.iterrows():
    log, final_action = run_advanced_agent(customer)
    
    print(f"Customer     : {customer['Customer_ID']}")
    print(f"Risk Score   : {log['Risk_Score']}")
    print(f"Decision     : {log['Initial_Decision']} → {final_action}")
    print(f"Reason       : {log['Reason'][:130]}...")
    if log['Guardrail_Applied'] != "No guardrail triggered":
        print(f"Guardrail    : {log['Guardrail_Applied']}")
    print(f"Tool Result  : {log['Tool_Result']}")
    print("-" * 90)

🚀 Testing Level 6 - Advanced Agent with Reflection & Tools

📱 SMS sent to CUST0362: 'Hi, we noticed a missed payment. Please pay at your earliest convenience.'
📝 Logged action 'SMS_REMINDER' for customer CUST0362
Customer     : CUST0362
Risk Score   : 55
Decision     : SMS_REMINDER → SMS_REMINDER
Reason       : Moderate risk. Previously took action 'SMS_REMINDER' because: Moderate risk. Previously took action 'SMS_REMINDER' because: Modera...
Tool Result  : SMS_SENT
------------------------------------------------------------------------------------------
📧 Payment plan offered to CUST0074: Flexible installments suggested.
📝 Logged action 'PAYMENT_PLAN' for customer CUST0074
Customer     : CUST0074
Risk Score   : 70
Decision     : PAYMENT_PLAN → PAYMENT_PLAN
Reason       : High risk. Previously took action 'PAYMENT_PLAN' because: High risk but recoverable. A structured payment plan combined with suppo...
Tool Result  : PAYMENT_PLAN_OFFERED
----------------------------------------------

##### 8: Final Summary & Save

In [14]:
# Code Block 8: Final Summary and Save
print("="*90)
print("LEVEL 6 - ADVANCED AGENTIC AI WITH REFLECTION & TOOLS")
print("="*90)
print(f"Total customers processed : {len(df)}")
print(f"Customers with memory     : {len(customer_memory)}")

print("\n✅ Level 6 Advanced Agent is complete!")
print("Key Capabilities Achieved:")
print("   • Natural reasoning with reflection on past actions")
print("   • Ethical guardrails")
print("   • Dynamic tool execution (SMS, Payment Plan, Escalate)")
print("   • Persistent memory across interactions")
print("   • Detailed logging")

# Save everything
os.makedirs('logs', exist_ok=True)

with open('logs/customer_memory_level6.json', 'w') as f:
    json.dump(customer_memory, f, indent=2)

df.to_excel('C:/Users/HP/Projects/Gen AI Powered Data Analytics/Final_Dataset_with_Agent_Level6_Advanced.xlsx', index=False)

print("\n📁 Files saved successfully!")
print("   • logs/customer_memory_level6.json")
print("   • C:/Users/HP/Projects/Gen AI Powered Data Analytics/Final_Dataset_with_Agent_Level6_Advanced.xlsx")
print("\n🎉 Congratulations! You now have a sophisticated Agentic AI prototype.")

LEVEL 6 - ADVANCED AGENTIC AI WITH REFLECTION & TOOLS
Total customers processed : 500
Customers with memory     : 500

✅ Level 6 Advanced Agent is complete!
Key Capabilities Achieved:
   • Natural reasoning with reflection on past actions
   • Ethical guardrails
   • Dynamic tool execution (SMS, Payment Plan, Escalate)
   • Persistent memory across interactions
   • Detailed logging

📁 Files saved successfully!
   • logs/customer_memory_level6.json
   • C:/Users/HP/Projects/Gen AI Powered Data Analytics/Final_Dataset_with_Agent_Level6_Advanced.xlsx

🎉 Congratulations! You now have a sophisticated Agentic AI prototype.
